# ThaiSum Manual Inference

Use this notebook after merging the LoRA adapter. It loads the merged model and lets you summarize Thai text interactively.

In [ ]:
import os
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
PROJECT_PATH = "."
MODEL_PATH = f"{PROJECT_PATH}/trained_model/qwen3-4b-thaisum"

In [ ]:
SYSTEM_PROMPT = (
    "คุณเป็นผู้ช่วยสรุปข้อความภาษาไทย\n"
    "สรุปให้กระชับ ชัดเจน และยึดตามข้อมูลในต้นฉบับเท่านั้น\n"
    "ห้ามเพิ่มข้อเท็จจริงใหม่ และตอบเป็นข้อความสรุปอย่างเดียว"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    device_map="auto",
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loaded model on: {device}")

In [ ]:
def postprocess_summary(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^(?:สรุป|summary)\s*[:：-]\s*", "", text, flags=re.IGNORECASE)
    return text.strip()


@torch.inference_mode()
def summarize(text: str, max_new_tokens: int = 256, temperature: float = 0.2) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text + "\n/no_think"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    model_inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        repetition_penalty=1.1,
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
    output_text = tokenizer.decode(output_ids, skip_special_tokens=True)
    return postprocess_summary(output_text)

In [ ]:
input_text = """วางบทความภาษาไทยที่ต้องการสรุปไว้ที่นี่"""
input_text

In [ ]:
result = summarize(input_text, max_new_tokens=256, temperature=0.2)
print(result)